In [10]:
import os
import json
import re
import glob
import pandas as pd

def extract_rows_from_slurm_ids(slurm_ids):
    data = []

    for slurm_id in slurm_ids:
        try:
            log_path = f"/n/home08/atong/projects/LMVF/logs/output_{slurm_id}.log"
            if not os.path.isfile(log_path):
                raise FileNotFoundError(f"Log file not found for slurm_id {slurm_id}")

            with open(log_path) as f:
                log_content = f.read()

            # Extract output directory from the log
            output_match = re.search(r"Output directory:\s+(.*)", log_content)
            if not output_match:
                raise ValueError(f"Output directory not found in log for slurm_id {slurm_id}")

            path = output_match.group(1).strip()
            cfg_path = os.path.join(path, "cfg")

            # Load config.json
            with open(os.path.join(cfg_path, "config.json")) as f:
                config = json.load(f)

            # Load max_new_tokens from model configs
            def get_max_new_tokens(filename):
                with open(os.path.join(cfg_path, filename)) as f:
                    return json.load(f)["model"]["max_new_tokens"]

            row = {
                "model": config["model"],
                "verifier_model": config["verifier_model"],
                "strict_verifier_model": config["strict_verifier_model"],
                "dataset": config["dataset"],
                "num_generations": config["num_generations"],
                "max_new_tokens_model": get_max_new_tokens("model_config.json"),
                "max_new_tokens_verifier": get_max_new_tokens("verifier_model_config.json"),
                "max_new_tokens_strict_verifier": get_max_new_tokens("strict_verifier_model_config.json"),
                "slurm_id": slurm_id,
                "path": path
            }

            # Aggregate accuracy_task_*.json
            accuracy_files = glob.glob(os.path.join(path, "accuracy_task_*.json"))
            if not accuracy_files:
                raise ValueError("No accuracy_task_*.json files found.")

            metric_sets = []
            acc_data_all = []
            for file in accuracy_files:
                with open(file) as f:
                    data_json = json.load(f)
                    acc_data_all.append(data_json)
                    metric_sets.append(set(data_json.keys()))

            if not all(metrics == metric_sets[0] for metrics in metric_sets):
                raise ValueError(f"Mismatch in accuracy metrics among files in {path}")

            metrics = metric_sets[0]
            for metric in metrics:
                total_correct = sum(d[metric]["correct_count"] for d in acc_data_all)
                total_eval = sum(d[metric]["total_eval"] for d in acc_data_all)
                accuracy = total_correct / total_eval if total_eval > 0 else None

                safe_metric = metric.replace("-", "").replace("@", "")
                row[f"{safe_metric}_correct_count"] = total_correct
                row[f"{safe_metric}_total_eval"] = total_eval
                row[f"{safe_metric}_accuracy"] = accuracy

            # Execution time
            time_matches = re.findall(r"Total execution time: (\d+):(\d+):(\d+)", log_content)
            times_seconds = [
                int(h)*3600 + int(m)*60 + int(s)
                for h, m, s in time_matches
            ]

            if times_seconds:
                avg_seconds = sum(times_seconds) / len(times_seconds)
                row["avg_total_exec_time_minutes"] = avg_seconds / 60
            else:
                row["avg_total_exec_time_minutes"] = None

            data.append(row)

        except Exception as e:
            print(f"Error processing slurm_id {slurm_id}\n{e}")
            continue

    return pd.DataFrame(data)


In [11]:
slurm_ids = ['10976441', '11066453', '11104922', '11109136', '11124153', '11124341', '11286669', '11324997', '11395474', '11406423', '11453020', '11453022']
rows = extract_rows_from_slurm_ids(slurm_ids)

In [12]:
rows

,model,verifier_model,strict_verifier_model,dataset,num_generations,max_new_tokens_model,max_new_tokens_verifier,max_new_tokens_strict_verifier,slurm_id,path,...,pass1_correct_count,pass1_total_eval,pass1_accuracy,bonmav_correct_count,bonmav_total_eval,bonmav_accuracy,avg_total_exec_time_minutes,pass4_correct_count,pass4_total_eval,pass4_accuracy
0,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,math,8,2048,32768,16,10976441,/n/netscratch/hankyang_lab/Lab/alex/LMVF/gemma...,...,214,500,0.428,286,500,0.572,149.233333,NaN,NaN,NaN
1,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,math,8,2048,8192,16,11066453,/n/netscratch/hankyang_lab/Lab/alex/LMVF/gemma...,...,200,500,0.400,277,500,0.554,77.583333,NaN,NaN,NaN
2,gemma-3-4b-it,gemma-3-4b-it,gemma-3-4b-it,math,8,2048,2048,16,11104922,/n/netscratch/hankyang_lab/Lab/alex/LMVF/gemma...,...,347,500,0.694,362,500,0.724,24.725000,NaN,NaN,NaN
3,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,math,8,2048,8192,2048,11109136,/n/netscratch/hankyang_lab/Lab/alex/LMVF/gemma...,...,186,500,0.372,242,500,0.484,86.033333,NaN,NaN,NaN
4,gemma-3-1b-it,gemma-3-1b-it,gemma-3-1b-it,math,8,2048,2048,16,11124153,/n/netscratch/hankyang_lab/Lab/alex/LMVF/gemma...,...,95,250,0.380,96,250,0.384,13.833333,NaN,NaN,NaN
5,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,math,8,8192,8192,2048,11124341,/n/netscratch/hankyang_lab/Lab/alex/LMVF/deeps...,...,356,500,0.712,395,500,0.790,76.316667,NaN,NaN,NaN
6,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-1b-it,math,8,2048,8192,16,11286669,/n/netscratch/hankyang_lab/Lab/alex/LMVF/gemma...,...,193,500,0.386,225,500,0.450,68.133333,NaN,NaN,NaN
7,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,math,8,2048,4096,16,11324997,/n/netscratch/hankyang_lab/Lab/alex/LMVF/gemma...,...,198,500,0.396,276,500,0.552,51.933333,NaN,NaN,NaN
8,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,math,8,2048,2048,16,11395474,/n/netscratch/hankyang_lab/Lab/alex/LMVF/gemma...,...,197,500,0.394,268,500,0.536,35.658333,NaN,NaN,NaN
9,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,math,4,2048,4096,16,11406423,/n/netscratch/hankyang_lab/Lab/alex/LMVF/gemma...,...,199,500,0.398,255,500,0.510,29.858333,280.0,500.0,0.560


In [17]:
selected_columns = [
    'model',
    'verifier_model',
    'strict_verifier_model',
    'max_new_tokens_model',
    'max_new_tokens_verifier',
    'max_new_tokens_strict_verifier',
    'pass1_accuracy',
    'bonmav_accuracy',
    'pass4_accuracy',
    'pass8_accuracy',
    'avg_total_exec_time_minutes'
]

filtered_df = rows[selected_columns]

In [18]:
filtered_df

,model,verifier_model,strict_verifier_model,max_new_tokens_model,max_new_tokens_verifier,max_new_tokens_strict_verifier,pass1_accuracy,bonmav_accuracy,pass4_accuracy,pass8_accuracy,avg_total_exec_time_minutes
0,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,32768,16,0.428,0.572,NaN,0.636,149.233333
1,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,8192,16,0.400,0.554,NaN,0.628,77.583333
2,gemma-3-4b-it,gemma-3-4b-it,gemma-3-4b-it,2048,2048,16,0.694,0.724,NaN,0.832,24.725000
3,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,2048,8192,2048,0.372,0.484,NaN,0.630,86.033333
4,gemma-3-1b-it,gemma-3-1b-it,gemma-3-1b-it,2048,2048,16,0.380,0.384,NaN,0.628,13.833333
5,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,8192,8192,2048,0.712,0.790,NaN,0.890,76.316667
6,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-1b-it,2048,8192,16,0.386,0.450,NaN,0.640,68.133333
7,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,4096,16,0.396,0.552,NaN,0.634,51.933333
8,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,2048,16,0.394,0.536,NaN,0.618,35.658333
9,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,4096,16,0.398,0.510,0.560,NaN,29.858333


In [19]:
self_verifier_filtered_df = filtered_df.iloc[[2,4,5]]

In [20]:
selected_columns_2 = [
    'model',
    'verifier_model',
    'strict_verifier_model',
    'max_new_tokens_model',
    'max_new_tokens_strict_verifier',
    'pass1_accuracy',
    'bonmav_accuracy',
    'pass8_accuracy',
    'avg_total_exec_time_minutes'
]
self_verifier_filtered_df[selected_columns_2]

,model,verifier_model,strict_verifier_model,max_new_tokens_model,max_new_tokens_strict_verifier,pass1_accuracy,bonmav_accuracy,pass8_accuracy,avg_total_exec_time_minutes
2,gemma-3-4b-it,gemma-3-4b-it,gemma-3-4b-it,2048,16,0.694,0.724,0.832,24.725000
4,gemma-3-1b-it,gemma-3-1b-it,gemma-3-1b-it,2048,16,0.380,0.384,0.628,13.833333
5,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,8192,2048,0.712,0.790,0.890,76.316667


In [17]:
diff_strict_filtered_df = filtered_df.iloc[[1,3,6]]

In [22]:
selected_columns_3 = [
    'model',
    'verifier_model',
    'strict_verifier_model',
    'max_new_tokens_strict_verifier',
    'pass1_accuracy',
    'bonmav_accuracy',
    'pass8_accuracy',
    'avg_total_exec_time_minutes'
]

In [23]:
diff_strict_filtered_df[selected_columns_3]

,model,verifier_model,strict_verifier_model,max_new_tokens_strict_verifier,pass1_accuracy,bonmav_accuracy,pass8_accuracy,avg_total_exec_time_minutes
1,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,16,0.400,0.554,0.628,77.583333
3,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,2048,0.372,0.484,0.630,86.033333
6,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-1b-it,16,0.386,0.450,0.640,68.133333
